In [1]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

application_token=os.getenv('PUSHOVER_TOKEN')
user_id=os.getenv('PUSHOVER_USER')
Open_weather_token=os.getenv('OPENWEATHER')

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(funcName)s - %(message)s"
)

In [2]:
pushover_url='https://api.pushover.net/1/messages.json'

#using push to get notification on your device about the weather 
def push(message):

    payload={
        'token':application_token,
        'user':user_id,
        'device': 'Soumil',
        'message': message
    }
    logging.info('Sending notification to device via Pushover')
    response=requests.post(url=pushover_url, data=payload)
    if response.status_code!=200:
        logging.error("Unable to send the notification")
        raise
    print(response)
    pass

 
# def get_coor(city: str):
#     url = 'https://nominatim.openstreetmap.org/search'

#     params = {
#         'q': city,
#         'format': 'json',
#         'limit': 1
#     }

#     headers = {
#         'User-Agent': 'weather-alert-bot'
#     }

#     response = requests.get(url, params=params, headers=headers, timeout=10)
#     if response.status_code!=200:
#         raise
#     data = response.json()

#     if not data:
#         return {'error': 'City not found'}
#     return data[0]

def weather_call(city):

    url='https://api.openweathermap.org/data/2.5/weather'

    parameters={
        'q': city,
        'appid': Open_weather_token,
        'units': 'metric'
    }
    logging.info('Calling Open Weather API')
    response=requests.get(url, params=parameters, timeout=10)
    if response.status_code!=200:
        logging.error('Unable to retrieve weather data from Open Weather')
        raise
    data=response.json()
    
    return data

In [3]:
weather_call_json = {
    "type": "function",
    "function": {  
        "name": "weather_call",
        "description": "Retrieves the current weather for a specified city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Name of the city (e.g., Delhi, London)"
                }
            },
            "required": ["city"]
        }
    }
}

In [13]:
prompt = """
You are a weather assistant inside a chatbot.

You receive weather data from an API through the conversation.

Your job is to:
• Explain the weather naturally.
• Answer user questions about how the weather may affect activities.

Users may ask questions such as:
- "Is it good weather for cricket?"
- "Should I carry an umbrella?"
- "Is it safe to go running?"
- "Will it feel hot outside?"

Use the weather data provided to reason about the answer.

For example:
• Rain → outdoor sports may not be ideal
• Extreme heat → recommend hydration
• Strong wind → caution for outdoor activities

Do not invent weather data.
Only interpret the weather information already provided.
You may give practical advice about activities based on the weather.

----------------------------------------

STRICT OUTPUT RULE (MANDATORY):

You must ALWAYS return a valid JSON object.

Do not output plain text.
Do not output explanations outside JSON.
Do not use markdown or code blocks.
Do not include extra commentary.

Your entire response must be a valid JSON object and nothing else.

The JSON structure must always be:

{
  "chatbot_response": "Natural explanation answering the user's question using the weather data.",
  "pushover_summary": "Short 1–2 sentence notification summarizing the key weather condition."
}

If you cannot answer the question, still return the response inside this JSON structure.
"""

In [22]:
from openai import OpenAI
import json
gemini_key=os.getenv('GOOGLE_GEMINI')
client=OpenAI(api_key=gemini_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") 

tools = [weather_call_json]

last_city=''

def chat(message, history):
    global last_city
    last_city1=last_city

    logging.info(f'Answering to {message}')


    messages=[{'role': 'system', 'content': prompt}]

    if last_city:
        messages.append({
            "role": "system",
            "content": f"The current city being discussed is {last_city}. If the user asks a follow-up weather question without specifying a city, assume they mean this city."})

    messages += history
    messages.append({'role':'user','content':message})
    
    condition=True  

    while condition:
        
        try:
            logging.info('calling LLM')
            llm_tool=client.chat.completions.create(
                model="gemini-2.5-flash-lite",
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )
            
            assistant_message = llm_tool.choices[0].message   
            logging.info(f"{assistant_message}")   
        
            messages.append({'role': 'assistant', 'content' : assistant_message.content})

            if assistant_message.tool_calls:

                tool_calls=llm_tool.choices[0].message.tool_calls

                for tool_call in tool_calls:
                    
                    messages.append({"role":"assistant","content":"Calling weather tool"})

                    tool_name=tool_call.function.name
                    tool_id=tool_call.id

                    function=globals().get(tool_name)
                
                    arguments=json.loads(tool_call.function.arguments)

                    city = arguments.get("city")

                    if city:
                        last_city = city
                    else:
                        if last_city:
                            arguments["city"] = last_city
                        else:
                            return "Please specify a city."
            

                    logging.info(f"calling function {tool_name}")
                    tool_response=function(**arguments)

                    tool_response = {
                        "role": "user",
                        "content": f"Weather API result: {json.dumps(tool_response)}"}

                    messages.append(tool_response)
        
            else:
                messages.append({
                "role": assistant_message.role,
                "content": assistant_message.content})
                condition = False

        except Exception as e:
            logging.error(e)
            raise

    try:
        llm_response=json.loads(llm_tool.choices[0].message.content)
    
        chatbot_response=llm_response["chatbot_response"]
        push_notification=llm_response["pushover_summary"]
        if last_city1!=last_city:
            push(push_notification)
    
        return chatbot_response

    except:
        return llm_tool.choices[0].message.content

In [ ]:
import gradio as gr
gr.ChatInterface(chat).launch()

2026-03-07 16:58:20,999 - INFO - _send_single_request - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-03-07 16:58:21,066 - INFO - _send_single_request - HTTP Request: GET http://127.0.0.1:7867/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-07 16:58:21,091 - INFO - _send_single_request - HTTP Request: HEAD http://127.0.0.1:7867/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


2026-03-07 16:58:21,337 - INFO - _send_single_request - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-03-07 16:58:22,149 - INFO - _send_single_request - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-07 16:58:32,341 - INFO - chat - Answering to How is the weather in New Delhi?
2026-03-07 16:58:32,341 - INFO - chat - calling LLM
2026-03-07 16:58:33,599 - INFO - _send_single_request - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-07 16:58:33,600 - INFO - chat - ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-13611940590313665224', function=Function(arguments='{"city":"New Delhi"}', name='weather_call'), type='function')])
2026-03-07 16:58:33,600 - INFO - chat - calling

<Response [200]>


2026-03-07 16:58:56,183 - INFO - chat - Answering to Should i expect rain?
2026-03-07 16:58:56,186 - INFO - chat - calling LLM
2026-03-07 16:58:57,015 - INFO - _send_single_request - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-07 16:58:57,015 - INFO - chat - ChatCompletionMessage(content='No, you should not expect rain in New Delhi today. The weather is clear and there is no rain forecasted.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
